In [ ]:
import os
import subprocess
import sys

# Use Keras 2 (the API the course code was written against)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Ensure TF + Keras 2 compatibility package are installed
try:
    import tensorflow as tf
    import tf_keras
    print(f"TensorFlow {tf.__version__} (tf_keras {tf_keras.__version__})")
except ImportError:
    print("Installing TensorFlow + Keras 2 compatibility package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "tensorflow", "tf_keras"])
    print()
    print("=" * 60)
    print("Install complete.")
    print("Now: Runtime > Restart session, then re-run this cell.")
    print("=" * 60)


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
l0 = Dense(units=1, input_shape=[1])
model = Sequential([l0])
model.compile(optimizer='sgd', loss='mean_squared_error')

xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

model.fit(xs, ys, epochs=500)

print(model.predict(np.array([10.0])))
print("Here is what I learned: {}".format(l0.get_weights()))

In [ ]:
export_dir = 'saved_model/1'
tf.saved_model.save(model, export_dir)

In [ ]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

In [ ]:
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)

In [ ]:
# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

In [ ]:
to_predict = np.array([[10.0]], dtype=np.float32)
print(to_predict)
interpreter.set_tensor(input_details[0]['index'], to_predict)
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])
print(tflite_results)